# ============================================
# MODULE 1: TIME SERIES DATA HANDLING
# ============================================
#
# Learning Objectives:
# - Understand time series data structure and properties
# - Test for stationarity in power systems data
# - Analyze autocorrelation and partial autocorrelation
# - Perform seasonal decomposition of energy data
# - Handle time-based features and transformations
#
# Real-World Application:
# Power systems generate sequential time-dependent data. Understanding temporal
# patterns is critical for:
# - Load forecasting (predicting future energy demand)
# - Renewable energy integration (solar/wind prediction)
# - Grid stability analysis (detecting abnormal patterns)
# - Maintenance scheduling (identifying degradation trends)
# Time series analysis forms the foundation for accurate predictions in power systems.
#
# Estimated Time: 3-4 hours
# ============================================

## Section 1: Import Libraries and Setup

In [ ]:
# Import pandas for time series manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for visualizations
import matplotlib.pyplot as plt

# Import seaborn for enhanced plots
import seaborn as sns

# Import statsmodels for time series analysis
# statsmodels provides statistical models and tests
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss

# Import datetime utilities
from datetime import datetime, timedelta

# Import scipy for statistical functions
from scipy import stats

# Import warnings
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Section 2: Generate Time Series Power Systems Data

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 1 year of hourly data for comprehensive time series analysis
# 365 days × 24 hours = 8760 hourly measurements
n_records = 8760

# Create datetime index starting from January 1, 2023
# DateTime index is essential for time series operations in pandas
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Extract temporal components for pattern creation
hours = date_range.hour  # Hour of day (0-23)
day_of_week = date_range.dayofweek  # Day of week (0=Monday, 6=Sunday)
day_of_year = date_range.dayofyear  # Day of year (1-365)
month = date_range.month  # Month (1-12)

# Component 1: Base load (constant component)
# Represents minimum continuous load
base_load = 100  # MW

# Component 2: Daily pattern (24-hour cycle)
# Peak during afternoon (hour 14), minimum early morning (hour 3)
# Using sine wave for smooth transition
daily_pattern = 40 * np.sin((hours - 6) * np.pi / 12)

# Component 3: Weekly pattern (7-day cycle)
# Lower load on weekends compared to weekdays
weekly_multiplier = np.where(day_of_week < 5, 1.0, 0.85)

# Component 4: Seasonal pattern (annual cycle)
# Higher load in summer (cooling) and winter (heating)
# Using sine wave with period of 365 days
# Peak in summer (day 180) and winter (day 365/day 1)
seasonal_pattern = 25 * np.sin((day_of_year - 80) * 2 * np.pi / 365)

# Component 5: Trend (gradual increase over time)
# Represents load growth due to population/economic growth
# Small positive trend: 0.5 MW increase per month
trend = np.arange(n_records) * (0.5 / (30 * 24))  # 0.5 MW per month

# Component 6: Random noise (stochastic variations)
# Represents unpredictable variations in load
noise = np.random.normal(0, 5, n_records)

# Combine all components to create realistic load time series
load_mw = base_load + daily_pattern * weekly_multiplier + seasonal_pattern + trend + noise

# Ensure no negative values (physically impossible for load)
load_mw = np.maximum(load_mw, 0)

# Generate temperature data with seasonal variation
# Temperature affects load (cooling in summer, heating in winter)
temperature_c = 15 + 12 * np.sin((day_of_year - 80) * 2 * np.pi / 365) + np.random.normal(0, 3, n_records)

# Create DataFrame with DateTime index
# Setting timestamp as index enables time series operations
df = pd.DataFrame({
    'load_mw': load_mw,
    'temperature_c': temperature_c
}, index=date_range)

# The index is now a DatetimeIndex
df.index.name = 'timestamp'

print(f"Generated {len(df)} hourly records")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nDataFrame info:")
print(df.info())

## Section 3: Time Series Properties and Components

In [ ]:
# Display first and last records
print("First 10 records:")
print(df.head(10))

print("\nLast 10 records:")
print(df.tail(10))

In [ ]:
# Plot the complete time series
# Visualizing the full series reveals long-term patterns

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Complete Load Time Series (1 Year)', fontsize=16, fontweight='bold')

# Plot 1: Load time series
axes[0].plot(df.index, df['load_mw'], linewidth=0.5, color='blue', alpha=0.7)
axes[0].set_ylabel('Load (MW)', fontsize=12, fontweight='bold')
axes[0].set_title('Hourly Load Data', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Add rolling mean to show trend
# Rolling mean smooths out short-term fluctuations
# Window of 168 hours = 1 week
rolling_mean = df['load_mw'].rolling(window=168, center=True).mean()
axes[0].plot(df.index, rolling_mean, linewidth=2, color='red', 
             label='7-Day Moving Average', alpha=0.8)
axes[0].legend()

# Plot 2: Temperature time series
axes[1].plot(df.index, df['temperature_c'], linewidth=0.5, color='green', alpha=0.7)
axes[1].set_ylabel('Temperature (°C)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[1].set_title('Temperature Over Time', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add rolling mean
temp_rolling_mean = df['temperature_c'].rolling(window=168, center=True).mean()
axes[1].plot(df.index, temp_rolling_mean, linewidth=2, color='red', 
             label='7-Day Moving Average', alpha=0.8)
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 4: Stationarity Testing

Stationarity is a key property for many time series models.
A stationary time series has:
- Constant mean over time
- Constant variance over time
- Constant autocorrelation structure over time

In [ ]:
# Visual inspection for stationarity
# Plot mean and variance over time windows

# Split data into 12 monthly chunks
# Check if statistical properties remain constant
window_size = len(df) // 12  # Approximately 1 month

# Calculate rolling statistics
# Window of 720 hours = approximately 1 month
rolling_mean = df['load_mw'].rolling(window=720).mean()
rolling_std = df['load_mw'].rolling(window=720).std()

# Plot original series with rolling statistics
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Visual Stationarity Check: Load Time Series', fontsize=16, fontweight='bold')

# Plot 1: Original series with rolling mean
axes[0].plot(df.index, df['load_mw'], color='blue', linewidth=0.5, alpha=0.5, label='Original')
axes[0].plot(df.index, rolling_mean, color='red', linewidth=2, label='Rolling Mean (30 days)')
axes[0].set_ylabel('Load (MW)', fontsize=12, fontweight='bold')
axes[0].set_title('Original Series with Rolling Mean', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Rolling standard deviation
axes[1].plot(df.index, rolling_std, color='green', linewidth=2)
axes[1].set_ylabel('Standard Deviation', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[1].set_title('Rolling Standard Deviation (30 days)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- If rolling mean is constant, mean is stationary")
print("- If rolling std is constant, variance is stationary")
print("- Our series shows seasonal variation, indicating NON-stationarity")

In [ ]:
# Augmented Dickey-Fuller (ADF) Test for stationarity
# Statistical test for presence of unit root (non-stationarity)
# Null hypothesis: Series has a unit root (non-stationary)
# Alternative hypothesis: Series is stationary

def adf_test(series, name=''):
    """
    Perform Augmented Dickey-Fuller test for stationarity.
    
    Parameters:
    series: Time series data
    name: Name of the series for display
    """
    print(f"\nAugmented Dickey-Fuller Test: {name}")
    print("=" * 60)
    
    # Perform ADF test
    # autolag='AIC' automatically selects number of lags
    result = adfuller(series.dropna(), autolag='AIC')
    
    # Extract test results
    adf_statistic = result[0]
    p_value = result[1]
    n_lags = result[2]
    n_obs = result[3]
    critical_values = result[4]
    
    print(f"ADF Statistic: {adf_statistic:.6f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Number of lags used: {n_lags}")
    print(f"Number of observations: {n_obs}")
    print("\nCritical Values:")
    for key, value in critical_values.items():
        print(f"  {key}: {value:.3f}")
    
    # Interpret results
    print("\nInterpretation:")
    if p_value < 0.05:
        print(f"Result: REJECT null hypothesis (p-value = {p_value:.6f} < 0.05)")
        print("Conclusion: Series is STATIONARY")
    else:
        print(f"Result: FAIL TO REJECT null hypothesis (p-value = {p_value:.6f} >= 0.05)")
        print("Conclusion: Series is NON-STATIONARY")
    
    return adf_statistic, p_value

# Test original load series
adf_test(df['load_mw'], 'Load (MW)')

In [ ]:
# KPSS Test (Kwiatkowski-Phillips-Schmidt-Shin)
# Alternative stationarity test
# Null hypothesis: Series is stationary
# Alternative hypothesis: Series has a unit root (non-stationary)

def kpss_test(series, name=''):
    """
    Perform KPSS test for stationarity.
    
    Parameters:
    series: Time series data
    name: Name of the series for display
    """
    print(f"\nKPSS Test: {name}")
    print("=" * 60)
    
    # Perform KPSS test
    # regression='c' tests for level stationarity
    result = kpss(series.dropna(), regression='c', nlags='auto')
    
    kpss_statistic = result[0]
    p_value = result[1]
    n_lags = result[2]
    critical_values = result[3]
    
    print(f"KPSS Statistic: {kpss_statistic:.6f}")
    print(f"P-value: {p_value:.6f}")
    print(f"Number of lags used: {n_lags}")
    print("\nCritical Values:")
    for key, value in critical_values.items():
        print(f"  {key}: {value:.3f}")
    
    # Interpret results
    print("\nInterpretation:")
    if p_value < 0.05:
        print(f"Result: REJECT null hypothesis (p-value = {p_value:.6f} < 0.05)")
        print("Conclusion: Series is NON-STATIONARY")
    else:
        print(f"Result: FAIL TO REJECT null hypothesis (p-value = {p_value:.6f} >= 0.05)")
        print("Conclusion: Series is STATIONARY")
    
    return kpss_statistic, p_value

# Test original load series
kpss_test(df['load_mw'], 'Load (MW)')

print("\n" + "=" * 60)
print("Note: ADF and KPSS tests have opposite null hypotheses.")
print("Use both tests together for robust stationarity assessment.")
print("=" * 60)

## Section 5: Making Series Stationary (Differencing)

In [ ]:
# Apply first-order differencing to remove trend
# Differencing: y'(t) = y(t) - y(t-1)
# This removes linear trends and can stabilize mean

# First difference (removes trend)
df['load_diff1'] = df['load_mw'].diff()

# Plot original vs differenced series
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Differencing to Achieve Stationarity', fontsize=16, fontweight='bold')

# Plot 1: Original series
axes[0].plot(df.index, df['load_mw'], linewidth=0.8, color='blue')
axes[0].set_ylabel('Load (MW)', fontsize=12, fontweight='bold')
axes[0].set_title('Original Series (Non-Stationary)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: First-differenced series
axes[1].plot(df.index, df['load_diff1'], linewidth=0.8, color='green')
axes[1].set_ylabel('First Difference', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[1].set_title('First Differenced Series', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add horizontal line at zero
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)

plt.tight_layout()
plt.show()

print("First differencing applied")

In [ ]:
# Test stationarity of differenced series
adf_test(df['load_diff1'], 'First Differenced Load')
kpss_test(df['load_diff1'], 'First Differenced Load')

In [ ]:
# Seasonal differencing to remove seasonal patterns
# For hourly data with daily pattern, lag = 24 hours
# y'(t) = y(t) - y(t-24)

df['load_seasonal_diff'] = df['load_mw'].diff(24)

# Plot seasonal differencing result
plt.figure(figsize=(16, 6))
plt.plot(df.index, df['load_seasonal_diff'], linewidth=0.8, color='purple')
plt.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
plt.ylabel('Seasonal Difference (lag=24)', fontsize=12, fontweight='bold')
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.title('Seasonally Differenced Load (24-hour lag)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Test stationarity
adf_test(df['load_seasonal_diff'], 'Seasonal Differenced Load (24h)')

## Section 6: Autocorrelation Analysis

In [ ]:
# Autocorrelation Function (ACF)
# Measures correlation between series and its lagged values
# ACF(k) = correlation between y(t) and y(t-k)

# Plot ACF for load data
# lags=168 shows correlations up to 1 week (168 hours)
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Autocorrelation Analysis: Load Data', fontsize=16, fontweight='bold')

# ACF plot
# alpha=0.05 shows 95% confidence interval
# Lags outside confidence band are statistically significant
plot_acf(df['load_mw'].dropna(), lags=168, ax=axes[0], alpha=0.05)
axes[0].set_title('Autocorrelation Function (ACF) - Up to 1 Week', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lag (hours)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('ACF', fontsize=11, fontweight='bold')

# Partial Autocorrelation Function (PACF)
# Measures direct correlation at each lag, removing indirect effects
# PACF is useful for identifying AR order in ARIMA models
plot_pacf(df['load_mw'].dropna(), lags=168, ax=axes[1], alpha=0.05)
axes[1].set_title('Partial Autocorrelation Function (PACF) - Up to 1 Week', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lag (hours)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('PACF', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Strong peaks at lag 24, 48, 72... indicate daily seasonality")
print("- Strong peak at lag 168 indicates weekly seasonality")
print("- Slowly decaying ACF suggests non-stationarity or strong persistence")

In [ ]:
# Calculate numerical ACF values for specific lags of interest
# This helps quantify correlation at key time intervals

# Calculate ACF for first 168 lags (1 week)
acf_values = acf(df['load_mw'].dropna(), nlags=168)

# Extract correlations at key lags
key_lags = [1, 24, 48, 72, 168]  # 1h, 1d, 2d, 3d, 1week

print("Autocorrelation at Key Lags:")
print("=" * 60)
for lag in key_lags:
    if lag < len(acf_values):
        print(f"Lag {lag:3d} ({lag:3d} hours / {lag/24:5.1f} days): {acf_values[lag]:.4f}")

print("\nNote: Values close to 1.0 indicate strong correlation")
print("      Values close to 0.0 indicate no correlation")
print("      Negative values indicate inverse correlation")

## Section 7: Seasonal Decomposition

In [ ]:
# Decompose time series into components:
# 1. Trend: Long-term progression
# 2. Seasonal: Repeating patterns
# 3. Residual: Random fluctuations

# Perform seasonal decomposition
# model='additive': assumes components add to form original series
# y(t) = Trend(t) + Seasonal(t) + Residual(t)
# period=24: Daily seasonality (24 hours)
decomposition = seasonal_decompose(df['load_mw'], model='additive', period=24)

# Extract components
trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(16, 14))
fig.suptitle('Seasonal Decomposition: Load Time Series (Daily Pattern)', 
             fontsize=16, fontweight='bold')

# Plot 1: Original series
axes[0].plot(df.index, df['load_mw'], linewidth=0.8, color='blue')
axes[0].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0].set_title('Original Series', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Trend component
axes[1].plot(df.index, trend, linewidth=1.5, color='red')
axes[1].set_ylabel('Trend', fontsize=11, fontweight='bold')
axes[1].set_title('Trend Component (Long-term Pattern)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Plot 3: Seasonal component
axes[2].plot(df.index, seasonal, linewidth=0.8, color='green')
axes[2].set_ylabel('Seasonal', fontsize=11, fontweight='bold')
axes[2].set_title('Seasonal Component (24-hour Pattern)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Plot 4: Residual component
axes[3].plot(df.index, residual, linewidth=0.5, color='purple', alpha=0.7)
axes[3].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[3].set_ylabel('Residual', fontsize=11, fontweight='bold')
axes[3].set_xlabel('Date', fontsize=11, fontweight='bold')
axes[3].set_title('Residual Component (Random Fluctuations)', fontsize=12, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Seasonal decomposition complete")

In [ ]:
# Analyze seasonal component in detail
# Extract one complete seasonal cycle (24 hours)

# Get first 24 hours of seasonal component
seasonal_pattern = seasonal[:24]

# Plot seasonal pattern
plt.figure(figsize=(14, 6))
plt.plot(range(24), seasonal_pattern, marker='o', markersize=8, 
         linewidth=2, color='green', label='Daily Seasonal Pattern')
plt.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
plt.xlabel('Hour of Day', fontsize=12, fontweight='bold')
plt.ylabel('Seasonal Component (MW)', fontsize=12, fontweight='bold')
plt.title('Daily Seasonal Pattern (Repeating 24-Hour Cycle)', fontsize=14, fontweight='bold')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nSeasonal Pattern Statistics:")
print(f"Maximum seasonal effect: {seasonal_pattern.max():.2f} MW at hour {seasonal_pattern.idxmax().hour}")
print(f"Minimum seasonal effect: {seasonal_pattern.min():.2f} MW at hour {seasonal_pattern.idxmin().hour}")
print(f"Peak-to-peak seasonal variation: {seasonal_pattern.max() - seasonal_pattern.min():.2f} MW")

In [ ]:
# Analyze residuals
# Residuals should be random (white noise) if decomposition is good

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Residual Analysis', fontsize=16, fontweight='bold')

# Plot 1: Histogram of residuals
# Should be approximately normal distribution
axes[0].hist(residual.dropna(), bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[0].set_xlabel('Residual Value', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Residuals', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Add normal distribution overlay
mu, std = residual.dropna().mean(), residual.dropna().std()
x = np.linspace(residual.dropna().min(), residual.dropna().max(), 100)
axes[0].plot(x, stats.norm.pdf(x, mu, std) * len(residual.dropna()) * 
             (residual.dropna().max() - residual.dropna().min()) / 50,
             'r-', linewidth=2, label='Normal Distribution')
axes[0].legend()

# Plot 2: Q-Q plot (quantile-quantile plot)
# Checks if residuals follow normal distribution
# Points should fall on diagonal line if normally distributed
stats.probplot(residual.dropna(), dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: Residuals vs Normal Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Residual Statistics:")
print(f"Mean: {residual.dropna().mean():.4f} (should be close to 0)")
print(f"Std Dev: {residual.dropna().std():.4f}")
print(f"Min: {residual.dropna().min():.4f}")
print(f"Max: {residual.dropna().max():.4f}")

## Section 8: Time-Based Feature Engineering

In [ ]:
# Create time-based features from datetime index
# These features capture temporal patterns for machine learning models

# Extract hour of day (0-23)
df['hour'] = df.index.hour

# Extract day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df.index.dayofweek

# Extract month (1-12)
df['month'] = df.index.month

# Extract day of year (1-365)
df['day_of_year'] = df.index.dayofyear

# Create binary weekend indicator
# 1 for Saturday/Sunday, 0 for weekdays
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

# Create categorical time of day
# Useful for capturing different load patterns during different periods
def time_of_day(hour):
    if 0 <= hour < 6:
        return 'Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'

df['time_of_day'] = df['hour'].apply(time_of_day)

# Create cyclical features using sine/cosine transformation
# This preserves the cyclical nature of time (hour 23 is close to hour 0)

# Hour as cyclical feature
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Day of week as cyclical feature
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Month as cyclical feature
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print("Time-based features created:")
print(df[['load_mw', 'hour', 'day_of_week', 'month', 'is_weekend', 
          'time_of_day', 'hour_sin', 'hour_cos']].head(10))

In [ ]:
# Visualize cyclical encoding
# Show how sine/cosine encoding preserves cyclical relationships

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cyclical Feature Encoding', fontsize=16, fontweight='bold')

# Plot 1: Hour encoding as polar plot
# This clearly shows the cyclical nature
hours_unique = np.arange(24)
hour_sin = np.sin(2 * np.pi * hours_unique / 24)
hour_cos = np.cos(2 * np.pi * hours_unique / 24)

axes[0].scatter(hour_sin, hour_cos, c=hours_unique, cmap='viridis', s=200)
for i, hour in enumerate(hours_unique):
    axes[0].annotate(f'{hour}h', (hour_sin[i], hour_cos[i]), 
                     fontsize=9, ha='center', va='center', color='white', fontweight='bold')
axes[0].set_xlabel('Hour Sin', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Hour Cos', fontsize=12, fontweight='bold')
axes[0].set_title('Hour of Day (Cyclical Encoding)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Add circle to show cyclical nature
circle = plt.Circle((0, 0), 1, fill=False, color='red', linestyle='--', linewidth=2)
axes[0].add_patch(circle)

# Plot 2: Month encoding
months_unique = np.arange(1, 13)
month_sin = np.sin(2 * np.pi * months_unique / 12)
month_cos = np.cos(2 * np.pi * months_unique / 12)

axes[1].scatter(month_sin, month_cos, c=months_unique, cmap='coolwarm', s=200)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for i, month in enumerate(month_names):
    axes[1].annotate(month, (month_sin[i], month_cos[i]), 
                     fontsize=9, ha='center', va='center', color='white', fontweight='bold')
axes[1].set_xlabel('Month Sin', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Month Cos', fontsize=12, fontweight='bold')
axes[1].set_title('Month (Cyclical Encoding)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# Add circle
circle = plt.Circle((0, 0), 1, fill=False, color='red', linestyle='--', linewidth=2)
axes[1].add_patch(circle)

plt.tight_layout()
plt.show()

print("\nCyclical encoding benefits:")
print("- Hour 23 and Hour 0 are now represented as close points")
print("- December and January are now close (unlike using month=12 and month=1)")
print("- Preserves natural cyclical relationships for ML models")

## What This Means for Electrical Engineers

### Industry Relevance:

1. **Load Forecasting**: Time series analysis is the foundation of:
   - Day-ahead load forecasting for market operations
   - Week-ahead forecasting for unit commitment
   - Long-term forecasting for capacity planning

2. **Renewable Energy Integration**:
   - Solar generation follows strong daily cycles
   - Wind patterns have autocorrelation structure
   - Understanding these patterns improves grid stability

3. **Equipment Health Monitoring**:
   - Trend detection identifies gradual degradation
   - Anomaly detection in residuals flags equipment issues
   - Seasonal patterns help schedule maintenance

4. **Grid Operations**:
   - Stationarity tests help validate forecasting models
   - Autocorrelation reveals persistence in demand
   - Seasonal decomposition separates controllable vs. weather-driven load

5. **Energy Markets**:
   - Price forecasting uses time series techniques
   - Cyclical patterns inform trading strategies
   - Trend analysis guides long-term contracts

### Key Takeaways:

- **Stationarity matters**: Many forecasting models assume stationarity
- **Differencing is powerful**: Simple technique to achieve stationarity
- **Autocorrelation reveals memory**: Past values influence future values
- **Decomposition provides insights**: Separating components aids understanding
- **Cyclical encoding is essential**: Preserves temporal relationships for ML

### Common Mistakes:

- Ignoring non-stationarity when building forecasting models
- Using linear time features (hour=0,1,2...) instead of cyclical encoding
- Over-differencing (removing too much information)
- Assuming all patterns are seasonal (some are stochastic)
- Not accounting for multiple seasonal patterns (daily + weekly + annual)

### Pro Tips:

- Always test for stationarity before modeling
- Use both ADF and KPSS tests (they complement each other)
- Plot ACF/PACF to understand temporal structure
- Consider multiple seasonal periods in power systems (24h, 168h, 8760h)
- Use cyclical encoding for time features in machine learning
- Residuals should look like white noise after good decomposition

### Next Steps:

In the next notebook (Feature Engineering), we'll create domain-specific features for power systems including lag features, rolling statistics, and electrical parameter transformations.